# Bukti GUI Orange 3.40.0: PCA iris dan heart_disease

Bukti dari GUI ada di `evidence/pca/` (setting dan daftar file: `evidence/pca/README.md`).
Aturan uji dan toleransi: `docs/orange-pca-internals.md`. Kode pembanding:
`tests/evidence_io.py` dan `tests/pca_compare.py` (sama dengan yang dipakai pytest).

**Alur bukti**

1. **GUI = skrip, bit-identik.** `.pkl` (skor, Save Data) dan `.npy` (komponen, Python
   Script) dari GUI dibandingkan dengan PCA Orange lewat skrip (projector widget,
   Normalize ON, dataset bawaan yang sama). Aturan uji spec: `np.allclose` default
   (rtol=1e-5, atol=1e-8); komponen seri (heart PC2, PC3) memakai satu tanda s per
   komponen untuk komponen dan skornya; ruang nol heart (PC19-25) hanya skor, atol=1e-8,
   rtol=0. Hasil yang diharapkan: selisih maks 0.0.
2. **CSV hanya pembulatan `%g`.** CSV dari Save Data dibandingkan dengan `.pkl`/`.npy`
   dengan rtol=1e-5, atol=0. `%g` menulis 6 digit signifikan, jadi selisih relatif
   maks harus ≤ 5e-6.
3. **Skrip jadi acuan zestcore.** Karena GUI dan skrip identik, tes zestcore cukup
   membandingkan dengan PCA Orange lewat skrip; GUI tidak perlu dibuka lagi.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import sklearn
import Orange

print("Python      ", sys.version.split()[0])
print("Orange      ", Orange.__version__)
print("NumPy       ", np.__version__)
print("scikit-learn", sklearn.__version__)

# Notebook dijalankan dari notebooks/; semua path relatif ke root repo.
ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "jalankan notebook dari folder notebooks/"
sys.path.insert(0, str(ROOT / "tests"))

# DeprecationWarning dari orangecanvas (paket Orange) saat import OWPCA; bukan dari kode ini.
warnings.filterwarnings("ignore", category=DeprecationWarning, module="orangecanvas")

Python       3.12.12
Orange       3.40.0
NumPy        2.1.3
scikit-learn 1.5.2


In [2]:
import pandas as pd
from Orange.data import Table

from evidence_io import (read_components_csv, read_components_npy,
                         read_scores_csv, read_scores_pkl)
from pca_compare import compare, compare_csv, orange_widget_pca, zero_variance_count

CASES = [("iris", "iris_pca_norm-on_4pc"),
         ("heart_disease", "heart_pca_norm-on_25pc")]


def pcs(rows):
    ks = [r["pc"] for r in rows]
    return f"PC{min(ks)}-{max(ks)}" if len(ks) > 1 else f"PC{ks[0]}"


summary, detail = [], []
for dataset, prefix in CASES:
    table = Table(dataset)
    n_null = zero_variance_count(table)
    ref_comp, ref_scores = orange_widget_pca(table)
    rows = compare(ref_comp, ref_scores,
                   read_components_npy(f"{prefix}_components.npy"),
                   read_scores_pkl(f"{prefix}_scores.pkl"), n_null)
    detail += [{"dataset": dataset, **r} for r in rows]
    n_nonnull = len(ref_comp) - n_null
    groups = [("GUI .npy vs skrip", "komponen", lambda r: True),
              ("GUI .pkl vs skrip", "skor", lambda r: r["pc"] <= n_nonnull),
              ("GUI .pkl vs skrip (ruang nol)", "skor", lambda r: r["pc"] > n_nonnull)]
    for name, jenis, keep in groups:
        g = [r for r in rows if r["jenis"] == jenis and keep(r)]
        if not g:
            continue
        summary.append({"dataset": dataset, "perbandingan": name, "jenis": jenis,
                        "PC": pcs(g),
                        "seri (s)": ", ".join(f"PC{r['pc']} ({r['s']:+d})"
                                              for r in g if r["seri"]) or "-",
                        "rtol": g[0]["rtol"], "atol": g[0]["atol"],
                        "allclose": all(r["allclose"] for r in g),
                        "selisih maks": max(r["maks"] for r in g),
                        "selisih rel. maks": None})
    if n_null:
        summary.append({"dataset": dataset, "perbandingan": "GUI .npy vs skrip (ruang nol)",
                        "jenis": "komponen", "PC": f"PC{n_nonnull + 1}-{len(ref_comp)}",
                        "seri (s)": "-", "rtol": None, "atol": None,
                        "allclose": "tidak dibandingkan", "selisih maks": None,
                        "selisih rel. maks": None})
    for name, jenis, csv, exact in [
            ("CSV vs .pkl", "skor", read_scores_csv(f"{prefix}_scores.csv"),
             read_scores_pkl(f"{prefix}_scores.pkl")),
            ("CSV vs .npy", "komponen", read_components_csv(f"{prefix}_components.csv"),
             read_components_npy(f"{prefix}_components.npy"))]:
        r = compare_csv(csv, exact)
        summary.append({"dataset": dataset, "perbandingan": name, "jenis": jenis,
                        "PC": f"PC1-{exact.shape[0] if jenis == 'komponen' else exact.shape[1]}",
                        "seri (s)": "-", "rtol": r["rtol"], "atol": r["atol"],
                        "allclose": r["allclose"], "selisih maks": r["maks"],
                        "selisih rel. maks": r["maks_rel"]})

pd.set_option("display.float_format", "{:.3g}".format)
summary = pd.DataFrame(summary)
summary

,dataset,perbandingan,jenis,PC,seri (s),rtol,atol,allclose,selisih maks,selisih rel. maks
0,iris,GUI .npy vs skrip,komponen,PC1-4,-,1e-05,1e-08,True,0,NaN
1,iris,GUI .pkl vs skrip,skor,PC1-4,-,1e-05,1e-08,True,0,NaN
2,iris,CSV vs .pkl,skor,PC1-4,-,1e-05,0,True,4.94e-06,4.82e-06
3,iris,CSV vs .npy,komponen,PC1-4,-,1e-05,0,True,4.94e-07,1.83e-06
4,heart_disease,GUI .npy vs skrip,komponen,PC1-18,"PC2 (+1), PC3 (+1)",1e-05,1e-08,True,0,NaN
5,heart_disease,GUI .pkl vs skrip,skor,PC1-18,"PC2 (+1), PC3 (+1)",1e-05,1e-08,True,0,NaN
6,heart_disease,GUI .pkl vs skrip (ruang nol),skor,PC19-25,-,0,1e-08,True,0,NaN
7,heart_disease,GUI .npy vs skrip (ruang nol),komponen,PC19-25,-,NaN,NaN,tidak dibandingkan,NaN,NaN
8,heart_disease,CSV vs .pkl,skor,PC1-25,-,1e-05,0,True,5e-06,4.95e-06
9,heart_disease,CSV vs .npy,komponen,PC1-25,-,1e-05,0,True,4.99e-07,4.7e-06


**Rincian per PC** (bukti GUI vs skrip). Kolom `s` hanya bisa -1 pada komponen seri.

In [3]:
pd.DataFrame(detail)

,dataset,jenis,pc,seri,s,rtol,atol,allclose,maks
0,iris,komponen,1,False,1,1e-05,1e-08,True,0
1,iris,skor,1,False,1,1e-05,1e-08,True,0
2,iris,komponen,2,False,1,1e-05,1e-08,True,0
3,iris,skor,2,False,1,1e-05,1e-08,True,0
4,iris,komponen,3,False,1,1e-05,1e-08,True,0
5,iris,skor,3,False,1,1e-05,1e-08,True,0
6,iris,komponen,4,False,1,1e-05,1e-08,True,0
7,iris,skor,4,False,1,1e-05,1e-08,True,0
8,heart_disease,komponen,1,False,1,1e-05,1e-08,True,0
9,heart_disease,skor,1,False,1,1e-05,1e-08,True,0


**Pemeriksaan alur bukti** (sel ini gagal bila salah satu klaim di atas tidak terpenuhi).

In [4]:
gui = summary[summary["perbandingan"].str.startswith("GUI")
              & (summary["allclose"] != "tidak dibandingkan")]
csv = summary[summary["perbandingan"].str.startswith("CSV")]
assert (gui["allclose"] == True).all() and (gui["selisih maks"] == 0.0).all()
assert (csv["allclose"] == True).all() and (csv["selisih rel. maks"] <= 5e-6).all()
print("1. GUI = skrip: selisih maks", gui["selisih maks"].max())
print("2. CSV vs .pkl/.npy: selisih rel. maks", f"{csv['selisih rel. maks'].max():.3g}", "<= 5e-6")

1. GUI = skrip: selisih maks 0.0
2. CSV vs .pkl/.npy: selisih rel. maks 4.95e-06 <= 5e-6
